In [8]:
from core.config import ConfigManager
from astropy.coordinates import SkyCoord
from pathlib import Path
import os
from core.logger import PipelineLogger
from core.directory_manager import DirectoryManager
from core.hdf5_handler import HDF5Handler
from core.map_tools import MapGenerator

In [9]:
config = ConfigManager('config.yaml')
method = config.get('fitting_procedure')
coordsys = config.get('coordinates.coord_sys', 'equatorial')
output_path = config.get('fitting.output_dir')
output_dir_name = config.get('fitting.fit_name')

# Test logger
logger = PipelineLogger('./logs')
logger.info("Pipeline test")

directory_manager = DirectoryManager(output_path, output_dir_name, logger=logger)
directory_manager.create_structure()

ra = config.get('coordinates.ra')
dec = config.get('coordinates.dec')
if ra is None or dec is None:
    l = config.get('coordinates.l')
    b = config.get('coordinates.b')
    if l is None or b is None:
        raise ValueError("Config must set either (ra, dec) or (l, b) for the ROI center")
    c = SkyCoord(l, b, frame='galactic', unit='deg')
    ra = float(c.icrs.ra.deg)
    dec = float(c.icrs.dec.deg)
    logger.info(f"Converted galactic coordinates (l={l}, b={b}) to equatorial (RA={ra}, Dec={dec})")
    config.set('coordinates.ra', ra)
    config.set('coordinates.dec', dec)


print(f"Fitting method: {method}")
print(f"Coordinate system: {coordsys}")
print(f"RA: {ra}")
print(f"Dec: {dec}")
print(f"Output path: {output_path}")
print(f"Output directory name: {output_dir_name}")

2026-08-24 10:17:51 - Pipeline - INFO - Pipeline logger initialized
2026-08-24 10:17:51 - Pipeline - INFO - Log level: INFO
2026-08-24 10:17:51 - Pipeline - INFO - Pipeline log: logs/pipeline_20260824_101751.log
2026-08-24 10:17:51 - Pipeline - INFO - Full log: logs/full_log_20260824_101751.log
2026-08-24 10:17:51 - Pipeline - INFO - Pipeline test
2026-08-24 10:17:51 - Pipeline - INFO - Initialized DirectoryManager at /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1
2026-08-24 10:17:51 - Pipeline - INFO - Directory structure created at /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1


Fitting method: Drips
Coordinate system: C
RA: 83
Dec: 22
Output path: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/
Output directory name: Run1


In [10]:
###Drips 
step_dir = directory_manager.get_step_results_dir('Step0-Allpoint-sources')

In [11]:
def _build_significance_map():
        """Build the significance map from count maps if coordinates.create_sig_map."""
        if not config.get('coordinates.create_sig_map', False):
            return None
        sig_map_path_cfe = config.get('coordinates.sig_map_path')
        if sig_map_path_cfe:
            sig_map_path = Path(sig_map_path_cfe)
        else:
            sig_map_path = directory_manager.get_datamap_dir() / "significance_map.fits"
            logger.info(f"Significance map already exists at {sig_map_path}, skipping generation")
            config.set("coordinates.sig_map_path", str(sig_map_path))
        if sig_map_path.exists():
            logger.info(f"Significance map already exists at {sig_map_path}, skipping generation")
            return sig_map_path
        
        # checkpoint.save_step('build_significance_map', 0, 'running', {})
        count_map_dir = config.get('coordinates.count_map_dir')
        image_bins = config.get('coordinates.image_bins')
        detector_response = config.get('coordinates.detector_response')

        fits_mapping = MapGenerator.find_fits_files_by_bins(count_map_dir, image_bins, logger=logger)
        if not fits_mapping:
            checkpoint.save_step('build_significance_map', 0, 'failed', {'error': 'no count-map FITS files found'})
            raise RuntimeError(f"No count-map FITS files found in {count_map_dir} for bins {image_bins}")
        ra = config.get('coordinates.ra')
        dec = config.get('coordinates.dec')
        if ra is None or dec is None:
            l = config.get('coordinates.l')
            b = config.get('coordinates.b')
            skycoord = SkyCoord(l, b, frame='galactic', unit='deg')
            ra = skycoord.icrs.ra.deg
            dec = skycoord.icrs.dec.deg
            logger.info(f"Converted galactic coordinates (l={l}, b={b}) to equatorial (RA={ra}, Dec={dec})")
            config.set('coordinates.ra', ra)
            config.set('coordinates.dec', dec) 
        
        output_path = MapGenerator.create_healpix_map(
            input_fits_files=list(fits_mapping.values()),
            energy_bins=list(fits_mapping.keys()),
            detector_response=detector_response,
            ra_center=float(config.get('coordinates.ra')),
            dec_center=float(config.get('coordinates.dec')),
            roi_x=float(10.0),
            roi_y=float(10.0),
            output_file=str(sig_map_path),
            logger=logger,
            pixi_manifest_path=config.get('alps.pixi_aerie_folder'),
        )


In [12]:
_build_significance_map()

2026-08-24 10:17:51 - Pipeline - INFO - Significance map already exists at /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/DataMapDir/significance_map.fits, skipping generation
2026-08-24 10:17:51 - Pipeline - INFO - Searching for FITS files in /Users/rishi/Documents/Analysis/data/fhitcountmaps/
2026-08-24 10:17:51 - Pipeline - INFO - Looking for bins: ['B2C0', 'B3C0', 'B4C0', 'B5C0', 'B6C0', 'B7C0', 'B8C0', 'B9C0', 'B10C0']
2026-08-24 10:17:51 - Pipeline - INFO - Found 9/9 energy bins
2026-08-24 10:17:51 - Pipeline - INFO - Creating HEALPix map from 9 FITS files
2026-08-24 10:17:51 - Pipeline - INFO - Energy bins: ['B2C0', 'B3C0', 'B4C0', 'B5C0', 'B6C0', 'B7C0', 'B8C0', 'B9C0', 'B10C0']
2026-08-24 10:17:51 - Pipeline - INFO - Executing HealpixSigFluxMap command:
2026-08-24 10:18:34 - Pipeline - INFO - HEALPix map created successfully: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProce

In [18]:
%load_ext autoreload
%autoreload 2
from drips_seeder import DRIPSSeeder
config = ConfigManager('config.yaml')
seeder = DRIPSSeeder(config, logger, directory_manager, step_path=str(step_dir))
drip_model_path = seeder.run()

2026-08-24 10:33:59 - Pipeline - INFO - Using significance map created: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/DataMapDir/significance_map.fits
2026-08-24 10:33:59 - Pipeline - INFO - Using galactic coordinates from config: (l=184.25262566216776, b=-6.287179548958748)
2026-08-24 10:33:59 - Pipeline - INFO - ROI size: 4.0° x 4.0°
2026-08-24 10:33:59 - Pipeline - INFO - Output directory for seeding: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step0-Allpoint-sources
2026-08-24 10:33:59 - Pipeline - INFO - Running seed model search in /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step0-Allpoint-sources
2026-08-24 10:33:59 - Pipeline - INFO - Loading HAWC data from /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
ROI center in Galactic Coordintes = 184.25262566216776, -6.287179548958748
Loading Galactic Map
Fits File loaded
Degrees per pixel: 0.005555555555555556 
  Map shape  : 1440 × 1440  |  pixel size: 0.0056°
  Sig range  : [-1637500000000000486439473119232.00, 311.27]
Peak intensity pixel location: (np.int64(810), np.int64(666))
Peak intensity sky location: <SkyCoord (Galactic): (l, b) in deg
    (184.54857275, -5.78153949)>
Peak intensity value: 311.26931404478034
Plotting contours [7, 9, 12, 13, 14, 15]
  Max significance 311.27sigma exceeds threshold 5.0sigma — proceeding with analysis.
  Image softly floored to -5sigma


Processing radius 0.25°:   0%|          | 0/4 [00:00<?, ?it/s]

Number of pixels corresponding to 0.25 smear radius = 45.00
Estimated background RMS: 0.0025402302952843877


Processing radius 0.30°:  25%|██▌       | 1/4 [00:06<00:18,  6.02s/it]

...

Raw blobs — point source: 2, extended: 1
Sources after 5$\sigma$ filtering: 3
Number of pixels corresponding to 0.50 smear radius = 90.00
Estimated background RMS: 0.0033270481966397704


Raw blobs — point source: 3, extended: 1
Sources after 5$\sigma$ filtering: 3
  PS  after all cuts:    2
  EXT after all cuts:    1
Plotting contours [7, 9, 12, 13, 14, 15]


Intensity Fraction of pixels greater than 5 sigma detection threshold = 100.0%
Larger blob coord = (np.float64(184.54857275384114), np.float64(-5.781539491143903))
  No smaller blobs overlapping larger blob — tagging as EXT
  Very close PS blob at (x=664, y=809, r=27.00 pixels) with sep=0.012° — tagging as EXT
PS flagged: 1
  Kept   — PS: 2  EXT: 0
  Removed— PS: 0  EXT: 1
  PS dedup: 2 → 2 kept, 0 removed
Final — PS kept: 2  EXT kept: 0
        PS removed: 0  EXT removed: 1
  Kept   — PS: 2  EXT: 0
  Removed— PS: 0  EXT: 1


2026-08-24 10:34:28 - Pipeline - INFO - Found 2 sources, 2 within original ROI
2026-08-24 10:34:28 - Pipeline - INFO - Results saved to: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step0-Allpoint-sources/filtered_sources.yaml
2026-08-24 10:34:28 - Pipeline - INFO - Results saved to: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step0-Allpoint-sources/filtered_sources.yaml


Converting blob at (x=664.0, y=809.0, r=27.00 pixels) to coord (RA=83.632°, Dec=22.011°) with radius 0.15°
Converting blob at (x=702.0, y=1250.0, r=27.00 pixels) to coord (RA=85.764°, Dec=23.488°) with radius 0.15°
Plotting contours [7, 9, 12, 13, 14, 15]
Model saved to: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step0-Allpoint-sources/curModel.model


In [19]:
%load_ext autoreload
%autoreload 2
import importlib
import source_fitter
import astromodels
importlib.reload(source_fitter)
logger.info('Starting source_fitter (DRIPS-seeded in-process fit)')
config = ConfigManager('config.yaml')
fit_output = source_fitter.run_joint_fit(drip_model_path, config, logger, directory_manager)

2026-08-24 10:34:32 - Pipeline - INFO - Starting source_fitter (DRIPS-seeded in-process fit)
2026-08-24 10:34:32 - Pipeline - INFO - Running joint fit on DRIPS seed model (/Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step0-Allpoint-sources/curModel.model) in /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit
2026-08-24 10:34:32 - Pipeline - INFO - Fitting model /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step0-Allpoint-sources/curModel.model in /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit  with compute_err=False, compute_TS=True, make_maps=True
2026-08-24 10:34:32 - Pipeline - INFO - Map tree: /Users/rishi/Documents/Analysis/data/maptree-fhit2pct-pass

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
['Source0', 'Source1']
83 22 15
Center of ROI  RA: 83.0000 Dec: 22.0000 Radius: 15.00
Source         RA:          Dec:           Distance to center
Source0          83.6319    22.0111   0.586 degrees
Source1          85.7644    23.4881   2.952 degrees


2026-08-24 10:34:47 - Pipeline - INFO - Running MLE without error estimation


Added Likelihood correction to: /Users/rishi/Documents/Analysis/aerie/.pixi/envs/threeml/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py
TOTAL =  200054.62190030178
TOTAL =  200054.62
CURRENT =  200054.62190030178
CURRENT =  200054.62


Best fit values:

...

2026-08-24 10:36:17 - Pipeline - INFO - Writing model map...
2026-08-24 10:36:17 - Pipeline - INFO - Fit Step1-JointFit: -logL=200054.622, AIC=400125.244 (1.76 min)
2026-08-24 10:36:17 - Pipeline - INFO - Converting HDF5 to FITS: residual_fit.hd5
2026-08-24 10:36:23 - Pipeline - INFO - Created FITS file: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit/fits/residual_binB2C0.fits.gz
2026-08-24 10:36:28 - Pipeline - INFO - Created FITS file: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit/fits/residual_binB3C0.fits.gz
2026-08-24 10:36:32 - Pipeline - INFO - Created FITS file: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit/fits/residual_binB4C0.fits.gz
2026-08-24 10:36:37 - Pipeline - INFO - Created FITS file: /Users/ris

Created FITS files: [PosixPath('/Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit/fits/residual_binB2C0.fits.gz'), PosixPath('/Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit/fits/residual_binB3C0.fits.gz'), PosixPath('/Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit/fits/residual_binB4C0.fits.gz'), PosixPath('/Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit/fits/residual_binB5C0.fits.gz'), PosixPath('/Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit/fits/residual_binB6C0.fits.gz'), PosixPath('/Users/rishi/Documents/Analysis/Sources/AstroImageDet

2026-08-24 10:37:22 - Pipeline - INFO - HEALPix map created successfully: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit/fits/residual.fits
2026-08-24 10:37:23 - Pipeline - INFO - Max value in residual map: nan


ROI center in Celestial Coordintes = None, None
Loading Celestial Map
Fits File loaded
Degrees per pixel: 1.0 
Peak intensity pixel location: (np.int64(0), np.int64(0))
Peak intensity sky location: <SkyCoord (ICRS): (ra, dec) in deg
    (nan, nan)>
Peak intensity value: nan
path.parent: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit


10:34:35 WARNING   The naima package is not available. Models    ]8;id=9358362;file:///Users/rishi/Documents/Analysis/aerie/.pixi/envs/threeml/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=9358363;file:///Users/rishi/Documents/Analysis/aerie/.pixi/envs/threeml/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py#43\43]8;;\
                  that depend on it will not be available                       
10:34:35 WARNING   The naima package is not available. Models    ]8;id=6678686;file:///Users/rishi/Documents/Analysis/aerie/.pixi/envs/threeml/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=6678687;file:///Users/rishi/Documents/Analysis/aerie/.pixi/envs/threeml/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py#43\43]8;;\
                  that depend on it will not be available                       
10:34:35 WARNI

In [22]:
from typing import List
def _as_list(value) -> List[str]:
    if value is None:
        return []
    return value if isinstance(value, list) else [value]


In [31]:
config = ConfigManager('config.yaml')
model = fit_output.model
baseline_log_like = fit_output.log_like
source_names = list(model.sources.keys())
alt_models = _as_list(config.get('fitting.alternate_spatial_models'))
alt_spectra = _as_list(config.get('fitting.alternate_spectral_models'))
coord_range = config.get('fitting.extended_source_coord_range', 1.0)
logger.info(f"Sources in the model: {source_names}")
logger.info(f"Baseline log-likelihood: {baseline_log_like}")
logger.info(f"Alternate spatial models: {alt_models}")
logger.info(f"Alternate spectral models: {alt_spectra}")

2026-08-24 10:44:07 - Pipeline - INFO - Sources in the model: ['Source0', 'Source1']
2026-08-24 10:44:07 - Pipeline - INFO - Baseline log-likelihood: 200054.62190030178
2026-08-24 10:44:07 - Pipeline - INFO - Alternate spatial models: ['Gaussian_on_sphere']
2026-08-24 10:44:07 - Pipeline - INFO - Alternate spectral models: ['Cutoff_powerlaw', 'Log_parabola']


In [ ]:
# # for sourcename in source_names:
# #     source = model.sources[sourcename]
# #     print(f"Checking source {source.name} for extension test")
# #     if source.name == 'URM':
# #         params = list(source.spatial_shape.parameters.items())
# #         # params = {k: v.value for k, v in params}
# #         print(f"Source {source.name} spatial shape: {params[0][1]}")
# #         params[0][1].free = False
# #         print(f"Source {source.name} spatial shape: {params[0][1].free}")

# for sourcename in source_names:
#     source = model.sources[sourcename]
#     print(f"Checking source {source.name} for extension test")
#     if source.name == 'URM':
#         params = list(source.spectrum.main._children.items())
#         spec_name, spec_func = params[0]
#         for pname, p in spec_func.parameters.items():
#             p.free = False
#         print(f"Source {source.name} spectrum: {spec_func}")
#         # params = {k: v.value for k, v in params}
#         # params[0][1].free = True
#         # print(f"Source {source.name} spatial shape: {params[0][1].free}")


In [ ]:
result = fit_output 
%load_ext autoreload
%autoreload 2
import importlib
import model_generator
import source_fitter

importlib.reload(source_fitter)
importlib.reload(model_generator)

alt_models = _as_list(config.get('fitting.alternate_spatial_models'))
if not alt_models:
    logger.info('No fitting.alternate_spatial_models configured; skipping extension test')

free_dbe = config.get('fitting.free_diffuse_norm', False)
threshold = config.get('likelihood_thresholds.extension_test', 16)
coord_range = config.get('fitting.extended_source_coord_range', 1.0)
runner = FitRunner(
    config_path=str(config.config_file),
    logger=logger,
    roi_template=config.get('roi.roi_template_path'),
)

model = fit_result.model
baseline_log_like = fit_result.log_like
source_names = list(model.sources.keys())
for source_name in source_names:
    if source_name == 'URM':
        logger.info(f'Skipping extension test for {source_name} (URM source)')
        continue
    other_sources = [n for n in model.sources.keys() if n != source_name]
    best_log_like = baseline_log_like
    best_model = model

    for alt_shape in alt_models:
        trial_model = model_generator.ModelGenerator.swap_spatial_shape(
            model, source_name, alt_shape, coord_range=coord_range, logger=logger,
        )
        model_generator.ModelGenerator.set_free(trial_model, other_sources, kind='spatial', free=False, free_diffuse=free_dbe, logger=logger)
        model_generator.ModelGenerator.set_free(trial_model, other_sources, kind='spectral', free=True, free_diffuse=free_dbe, logger=logger)

        step_name = f'Step2-{source_name}-Extension-{alt_shape}'
        step_dir = directory_manager.get_step_results_dir(step_name)
        model_file = model_generator.ModelGenerator.write_model_from_live(
            trial_model, str(directory_manager.get_model_file_path(step_name)), logger=logger,
        )
        trial_model.save("{1}/{0}.yml".format('curModel', step_dir), overwrite=True)
        model_generator.ModelGenerator.write_model_file_from_yaml("{1}/{0}.yml".format('curModel', step_dir), "{1}/{0}.model".format('curModel', step_dir), logger=logger)

        trial_result = runner.fit(
            model_file=str(model_file),
            step_dir=str(step_dir),
            compute_err=config.get('error_and_TS.error_extension', True),
            make_maps=False,
        )

        # logger.info(trial_model)
        # ModelGenerator.set_free(trial_model, other_sources, kind='spatial', free=False)
        # ModelGenerator.set_free(trial_model, other_sources, kind='spectral', free=False)

        # step_name = f'Step2-{source_name}-Extension-{alt_shape}'
        # step_dir = directory_manager.get_step_results_dir(step_name)
        # model_file = ModelGenerator.write_model_from_live(
        #     trial_model, str(directory_manager.get_model_file_path(step_name)), logger=logger,
        # )

In [42]:
result = fit_output 

%load_ext autoreload
%autoreload 2
import importlib
import model_generator
import source_fitter
importlib.reload(source_fitter)
importlib.reload(model_generator)
logger.info('Starting extension fit')


if config.get('fitting.run_extension_test', True):
    result_ext = source_fitter.run_extension_test(result, config, logger, directory_manager)

2026-08-24 11:06:55 - Pipeline - INFO - Starting extension fit
2026-08-24 11:06:55 - Pipeline - INFO - Diffuse background normalization status during spectrum test: True
2026-08-24 11:06:55 - Pipeline - INFO - Current best log-likelihood: 200054.622
2026-08-24 11:06:55 - Pipeline - INFO - Spatial shape of source Point_Source, Alternate model Gaussian_on_sphere
2026-08-24 11:06:55 - Pipeline - INFO - SOURCE Source1 : param ra -> FIXED (not in param_names)
2026-08-24 11:06:55 - Pipeline - INFO - SOURCE Source1 : param dec -> FIXED (not in param_names)
2026-08-24 11:06:55 - Pipeline - INFO - TESTING SPECTRAL
2026-08-24 11:06:55 - Pipeline - INFO - SOURCE Source1 : param K -> free=True
2026-08-24 11:06:55 - Pipeline - INFO - SOURCE Source1 : param piv -> FIXED (not in param_names)
2026-08-24 11:06:55 - Pipeline - INFO - SOURCE Source1 : param index -> free=True
2026-08-24 11:06:55 - Pipeline - INFO - Fitting model /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-Im

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
['Source1', 'Source0']
83 22 15
Center of ROI  RA: 83.0000 Dec: 22.0000 Radius: 15.00
Source         RA:          Dec:           Distance to center
Source1          85.7157    23.4480   2.893 degrees
Source0          83.6336    22.0119   0.588 degrees


2026-08-24 11:07:09 - Pipeline - INFO - Running MLE without error estimation


Added Likelihood correction to: /Users/rishi/Documents/Analysis/aerie/.pixi/envs/threeml/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py
TOTAL =  200484.35181024385
TOTAL =  200484.35
CURRENT =  200484.35181024385
CURRENT =  200484.35


Best fit values:

...

2026-08-24 11:08:02 - Pipeline - INFO - Fit results saved to /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step2-Source1-Extension-Gaussian_on_sphere/likelihoodResults.fits
2026-08-24 11:08:02 - Pipeline - INFO - Saving HAL output maps
2026-08-24 11:08:02 - Pipeline - INFO - SAVE A BIG MAP
2026-08-24 11:08:26 - Pipeline - INFO - Writing model map...
2026-08-24 11:08:27 - Pipeline - INFO - Fit Step2-Source1-Extension-Gaussian_on_sphere: -logL=199969.054, AIC=399948.108 (0.77 min)
2026-08-24 11:08:27 - Pipeline - INFO - Extension test Source1 -> Gaussian_on_sphere: delta_TS=171.14 (threshold 16)
2026-08-24 11:08:27 - Pipeline - INFO - delta_TS above threshold; computing per-source TS for Step2-Source1-Extension-Gaussian_on_sphere
2026-08-24 11:08:27 - Pipeline - INFO - Calculating TS for all sources in the model
2026-08-24 11:08:27 - Pipeline - INFO - Computing TS for source: Source0
2026-08-24 11:08:31 - Pipel

Added Likelihood correction to: /Users/rishi/Documents/Analysis/aerie/.pixi/envs/threeml/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py
TOTAL =  247602.32646054833
TOTAL =  247602.33
CURRENT =  247602.32646054833
CURRENT =  247602.33


2026-08-24 11:08:31 - Pipeline - INFO - TS for source Source1: 226.66692540573422
2026-08-24 11:08:31 - Pipeline - INFO - Accepted alternate spatial model Gaussian_on_sphere for Source1


Added Likelihood correction to: /Users/rishi/Documents/Analysis/aerie/.pixi/envs/threeml/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py
TOTAL =  200082.3874830244
TOTAL =  200082.39
CURRENT =  200082.3874830244
CURRENT =  200082.39


11:06:58 WARNING   The naima package is not available. Models    ]8;id=7016736;file:///Users/rishi/Documents/Analysis/aerie/.pixi/envs/threeml/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=7016737;file:///Users/rishi/Documents/Analysis/aerie/.pixi/envs/threeml/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py#43\43]8;;\
                  that depend on it will not be available                       
11:06:58 WARNING   The naima package is not available. Models    ]8;id=13346169;file:///Users/rishi/Documents/Analysis/aerie/.pixi/envs/threeml/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=13346170;file:///Users/rishi/Documents/Analysis/aerie/.pixi/envs/threeml/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py#43\43]8;;\
                  that depend on it will not be available                       
11:06:58 WAR

In [43]:
result_ext

FitResult(model=Model summary:

                  N
Point sources     1
Extended sources  1
Particle sources  0

Free parameters (5):
--------------------

                                         value min_value max_value  \
Source0.spectrum.main.Powerlaw.K           0.0       0.0       0.0   
Source0.spectrum.main.Powerlaw.index -2.587876      -3.0      -1.0   
Source1.Gaussian_on_sphere.sigma      1.006653      0.01       3.0   
Source1.spectrum.main.Powerlaw.K           0.0       0.0       0.0   
Source1.spectrum.main.Powerlaw.index -2.129917      -3.0      -1.0   

                                                unit  
Source0.spectrum.main.Powerlaw.K      keV-1 s-1 cm-2  
Source0.spectrum.main.Powerlaw.index                  
Source1.Gaussian_on_sphere.sigma                 deg  
Source1.spectrum.main.Powerlaw.K      keV-1 s-1 cm-2  
Source1.spectrum.main.Powerlaw.index                  

Fixed parameters (7):
(abridged. Use complete=True to see all fixed parameters)


Properties

In [44]:
%load_ext autoreload
%autoreload 2
import importlib
import model_generator
import source_fitter
importlib.reload(source_fitter)
importlib.reload(model_generator)
logger.info('Starting extension fit')


if config.get('fitting.run_spectrum_test', True):
    result_spectrum_test = source_fitter.run_spectrum_test(result_ext, config, logger, directory_manager)

2026-08-24 11:08:32 - Pipeline - INFO - Starting extension fit
2026-08-24 11:08:32 - Pipeline - INFO - Diffuse background normalization status during spectrum test: True
2026-08-24 11:08:32 - Pipeline - INFO - Swapping Source0 spectral shape from Powerlaw to Cutoff_powerlaw
2026-08-24 11:08:32 - Pipeline - INFO - New spectrum class: dict_keys(['K', 'piv', 'index', 'xc'])


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


2026-08-24 11:08:32 - Pipeline - INFO - Swapped Source0 spectral shape -> Cutoff_powerlaw
2026-08-24 11:08:32 - Pipeline - INFO - SOURCE Source1 : param lon0 -> FIXED (not in param_names)
2026-08-24 11:08:32 - Pipeline - INFO - SOURCE Source1 : param lat0 -> FIXED (not in param_names)
2026-08-24 11:08:32 - Pipeline - INFO - SOURCE Source1 : param sigma -> free=True
2026-08-24 11:08:32 - Pipeline - INFO - TESTING SPECTRAL
2026-08-24 11:08:32 - Pipeline - INFO - SOURCE Source1 : param K -> free=True
2026-08-24 11:08:32 - Pipeline - INFO - SOURCE Source1 : param piv -> FIXED (not in param_names)
2026-08-24 11:08:32 - Pipeline - INFO - SOURCE Source1 : param index -> free=True
2026-08-24 11:08:32 - Pipeline - INFO - Fitting model /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step3-Source0-Spectrum-Cutoff_powerlaw/curModel.model in /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProc

['Source0', 'Source1']
83 22 15
Center of ROI  RA: 83.0000 Dec: 22.0000 Radius: 15.00
Source         RA:          Dec:           Distance to center
Source0          83.6336    22.0119   0.588 degrees
Source1          85.7157    23.4480   2.893 degrees


2026-08-24 11:08:49 - Pipeline - INFO - Running MLE without error estimation


...

Values of -log(likelihood) at the minimum:

,-log(likelihood)
HAWC,199639.189245
total,199639.189245


Values of statistical measures:

,statistical measures
AIC,399292.378543
BIC,399380.289306


2026-08-24 11:10:53 - Pipeline - INFO - Fit results saved to /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step3-Source1-Spectrum-Log_parabola/likelihoodResults.fits
2026-08-24 11:10:53 - Pipeline - INFO - Fit Step3-Source1-Spectrum-Log_parabola: -logL=199639.189, AIC=399292.379 (0.67 min)
2026-08-24 11:10:53 - Pipeline - INFO - Spectrum test Source1 -> Log_parabola: delta_TS=-0.00 (threshold 16)
2026-08-24 11:10:53 - Pipeline - INFO - Rejected alternate spectral model Log_parabola for Source1; skipping TS computation
11:08:35 WARNING   The naima package is not available. Models    ]8;id=10518026;file:///Users/rishi/Documents/Analysis/aerie/.pixi/envs/threeml/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=10518027;file:///Users/rishi/Documents/Analysis/aerie/.pixi/envs/threeml/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.p

In [59]:
# result = fit_output 

%load_ext autoreload
%autoreload 2
import importlib
import model_generator
import source_fitter
importlib.reload(source_fitter)
importlib.reload(model_generator)
logger.info('Starting extension fit')


# if config.get('fitting.run_spectrum_test', True):
#     result_ext = source_fitter.run_spectrum_test(result_ext, config, logger, directory_manager)

if config.get('fitting.run_final_refit', True):
    final_result = source_fitter.run_final_refit(result_spectrum_test, config, logger, directory_manager)

2026-08-24 11:19:57 - Pipeline - INFO - Starting extension fit
2026-08-24 11:19:57 - Pipeline - INFO - SOURCE Source0 : param ra -> free=True
2026-08-24 11:19:57 - Pipeline - INFO - SOURCE Source0 : param dec -> free=True
2026-08-24 11:19:57 - Pipeline - INFO - SOURCE Source1 : param lon0 -> free=True
2026-08-24 11:19:57 - Pipeline - INFO - SOURCE Source1 : param lat0 -> free=True
2026-08-24 11:19:57 - Pipeline - INFO - SOURCE Source1 : param sigma -> free=True
2026-08-24 11:19:57 - Pipeline - INFO - TESTING SPECTRAL
2026-08-24 11:19:57 - Pipeline - INFO - SOURCE Source0 : param K -> free=True
2026-08-24 11:19:57 - Pipeline - INFO - SOURCE Source0 : param piv -> FIXED (not in param_names)
2026-08-24 11:19:57 - Pipeline - INFO - SOURCE Source0 : param index -> free=True
2026-08-24 11:19:57 - Pipeline - INFO - SOURCE Source0 : param xc -> free=True
2026-08-24 11:19:57 - Pipeline - INFO - TESTING SPECTRAL
2026-08-24 11:19:57 - Pipeline - INFO - SOURCE Source1 : param K -> free=True
2026-0

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
['Source0', 'Source1']
83 22 15
Center of ROI  RA: 83.0000 Dec: 22.0000 Radius: 15.00
Source         RA:          Dec:           Distance to center
Source0          83.6336    22.0119   0.588 degrees
Source1          85.7157    23.4480   2.893 degrees


2026-08-24 11:20:11 - Pipeline - INFO - Running MLE without error estimation


Added Likelihood correction to: /Users/rishi/Documents/Analysis/aerie/.pixi/envs/threeml/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py
TOTAL =  199637.47717685247
TOTAL =  199637.48
CURRENT =  199637.47717685247
CURRENT =  199637.48


Best fit values:

...

2026-08-24 11:20:59 - Pipeline - INFO - Writing model map...
2026-08-24 11:21:00 - Pipeline - INFO - Fit Step4-FinalRefit: -logL=199637.477, AIC=399294.954 (1.05 min)
2026-08-24 11:21:00 - Pipeline - INFO - Converting HDF5 to FITS: residual_fit.hd5
2026-08-24 11:21:05 - Pipeline - INFO - Created FITS file: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step4-FinalRefit/fits/residual_binB2C0.fits.gz
2026-08-24 11:21:10 - Pipeline - INFO - Created FITS file: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step4-FinalRefit/fits/residual_binB3C0.fits.gz
2026-08-24 11:21:14 - Pipeline - INFO - Created FITS file: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step4-FinalRefit/fits/residual_binB4C0.fits.gz
2026-08-24 11:21:19 - Pipeline - INFO - Created FITS file: /U

Created FITS files: [PosixPath('/Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step4-FinalRefit/fits/residual_binB2C0.fits.gz'), PosixPath('/Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step4-FinalRefit/fits/residual_binB3C0.fits.gz'), PosixPath('/Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step4-FinalRefit/fits/residual_binB4C0.fits.gz'), PosixPath('/Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step4-FinalRefit/fits/residual_binB5C0.fits.gz'), PosixPath('/Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step4-FinalRefit/fits/residual_binB6C0.fits.gz'), PosixPath('/Users/rishi/Documents/Analysis/Sources/Ast

2026-08-24 11:21:33 - Pipeline - INFO - No hotspots found in /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step4-FinalRefit/fits/residual.fits. Max value: nan
2026-08-24 11:21:33 - Pipeline - INFO - Max value in residual map: nan
2026-08-24 11:21:33 - Pipeline - INFO - Wrote fit summary: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step4-FinalRefit/fit_summary.json
2026-08-24 11:21:33 - Pipeline - INFO - Wrote fit summary CSV: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step4-FinalRefit/fit_summary.csv


Fits File loaded
Degrees per pixel: 1.0 
Peak intensity pixel location: (np.int64(0), np.int64(0))
Peak intensity sky location: <SkyCoord (ICRS): (ra, dec) in deg
    (nan, nan)>
Peak intensity value: nan
path.parent: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step4-FinalRefit
Saved figure to /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step4-FinalRefit/fitsMap.png


11:20:00 WARNING   The naima package is not available. Models    ]8;id=8010374;file:///Users/rishi/Documents/Analysis/aerie/.pixi/envs/threeml/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=8010375;file:///Users/rishi/Documents/Analysis/aerie/.pixi/envs/threeml/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py#43\43]8;;\
                  that depend on it will not be available                       
11:20:00 WARNING   The naima package is not available. Models    ]8;id=1954238;file:///Users/rishi/Documents/Analysis/aerie/.pixi/envs/threeml/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=1954239;file:///Users/rishi/Documents/Analysis/aerie/.pixi/envs/threeml/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py#43\43]8;;\
                  that depend on it will not be available                       
11:20:00 WARNI

In [62]:
final_result

FitResult(model=Model summary:

                  N
Point sources     1
Extended sources  1
Particle sources  0

Free parameters (10):
--------------------

                                                       value     min_value  \
Source1.Gaussian_on_sphere.lon0                    85.594052     84.715655   
Source1.Gaussian_on_sphere.lat0                    23.283622      22.44801   
Source1.Gaussian_on_sphere.sigma                    1.003542          0.01   
Source1.spectrum.main.Powerlaw.K                         0.0           0.0   
Source1.spectrum.main.Powerlaw.index               -2.169583          -3.0   
Source0.position.ra                                83.633547     80.631907   
Source0.position.dec                               22.011802     19.011117   
Source0.spectrum.main.Cutoff_powerlaw.K                  0.0           0.0   
Source0...index                                     -2.39128         -10.0   
Source0.spectrum.main.Cutoff_powerlaw.xc  26784414731.128834  1

In [61]:
# %load_ext autoreload
# %autoreload 2
# import importlib
# import model_generator
# import source_fitter
# import pipeline_helpers
# importlib.reload(source_fitter)
# importlib.reload(model_generator)
# importlib.reload(pipeline_helpers)
# source_fitter.check_hotspots(resmap, final_result, config, logger)